In [ ]:
# 1. 공식 레포지토리 클론 및 폴더 이동
import os
if not os.path.exists('Dinomaly_repo'):
    !git clone https://github.com/guojiajeremy/Dinomaly.git Dinomaly_repo

os.chdir('Dinomaly_repo')
print(f"현재 작업 디렉토리: {os.getcwd()}")

Cloning into 'Dinomaly_repo'...
remote: Enumerating objects: 346, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 346 (delta 52), reused 46 (delta 46), pack-reused 273 (from 1)
Receiving objects: 100% (346/346), 380.16 KiB | 8.26 MiB/s, done.
Resolving deltas: 100% (129/129), done.
현재 작업 디렉토리: /kaggle/working/Dinomaly_repo


In [ ]:
import pandas as pd

utils_path = 'utils.py'
with open(utils_path, 'r', encoding='utf-8') as f:
    content = f.read()

# append 로직을 pd.concat으로 변경
old_code = 'df = df.append({"pro": mean(pros), "fpr": fpr, "threshold": th}, ignore_index=True)'
new_code = 'new_row = pd.DataFrame([{"pro": mean(pros), "fpr": fpr, "threshold": th}]); df = pd.concat([df, new_row], ignore_index=True)'

if old_code in content:
    content = content.replace(old_code, new_code)
    with open(utils_path, 'w', encoding='utf-8') as f:
        f.write(content)
    print("✅ utils.py 패치 완료 (append -> pd.concat)")
else:
    print("ℹ️ 이미 패치가 적용되어 있거나 코드를 찾을 수 없습니다.")

# CUDA 디바이스 인덱스 패치 (cuda:1 -> cuda:0)
main_script = 'dinomaly_mvtec_uni.py'
with open(main_script, 'r', encoding='utf-8') as f:
    content = f.read()

if "device = 'cuda:1'" in content:
    content = content.replace("device = 'cuda:1'", "device = 'cuda:0' if torch.cuda.is_available() else 'cpu'")
    with open(main_script, 'w', encoding='utf-8') as f:
        f.write(content)
    print("✅ dinomaly_mvtec_uni.py 패치 완료 (cuda:1 -> cuda:0)")

✅ utils.py 패치 완료 (append -> pd.concat)
✅ dinomaly_mvtec_uni.py 패치 완료 (cuda:1 -> cuda:0)


In [ ]:
# 3. 필수 라이브러리 설치
!pip install ptflops timm colorama

In [ ]:
# 4. Sanity Check 실행 (bottle)
# DATA_PATH를 본인의 MVTec AD 데이터셋 경로로 수정하세요.
DATA_PATH = os.path.abspath("/kaggle/input/datasets/lithos0534/mvtec2/mvtec_anomaly_detection")

if not os.path.exists(DATA_PATH):
    print(f"❌ [경로 오류] 데이터셋이 {DATA_PATH} 에 존재하지 않습니다.")
else:
    # bottle 카테고리 단일 실행
    !python dinomaly_mvtec_uni.py --data_path {DATA_PATH} --save_name sanity_check_bottle

/kaggle/working/Dinomaly_repo/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/kaggle/working/Dinomaly_repo/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/kaggle/working/Dinomaly_repo/dinov2/layers/block.py:38: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
cuda:0
100%|████████████████████████████████████████| 330M/330M [00:08<00:00, 40.5MB/s]
train image number:3629
iter [226/10000], loss:0.2052
iter [452/10000], loss:0.0898
iter [678/10000], loss:0.0743
iter [904/10000], loss:0.0676
ite